**Analise de Dados**

In [0]:
import pyspark.sql.functions as F
import matplotlib.pyplot as plt

# ============================================================
# Tabelas Silver que serão avaliadas
# ============================================================

tables = [
    "silver_full",
    "silver_gols",
    "silver_cartoes",
    "silver_estatisticas"
]

print("=== ANÁLISE DE QUALIDADE — CAMADA SILVER ===")


# ============================================================
# 1) VERIFICAÇÃO DE VALORES NULOS
# ============================================================

print("\n===== 1) VALORES NULOS POR COLUNA =====")

for table in tables:
    df = spark.table(table)
    print(f"\n### Tabela: {table} ###")

    nulls_df = df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])
    
    display(nulls_df)



# ============================================================
# 2) VERIFICAÇÃO DE SCHEMA (TIPOS DE DADOS)
# ============================================================

print("\n===== 2) TIPOS DE DADOS (SCHEMA) =====")

for table in tables:
    print(f"\n### Schema da tabela: {table} ###")
    spark.table(table).printSchema()



# ============================================================
# 3) ANÁLISE DE DOMÍNIO — COLUNAS CATEGÓRICAS
# ============================================================

print("\n===== 3) DOMÍNIO DAS CATEGORIAS =====")

dominio_cols = {
    "silver_cartoes": ["cartao", "posicao"],
    "silver_gols": ["tipo_de_gol"],
    "silver_full": ["mandante", "visitante", "arena"]
}

for table, cols in dominio_cols.items():
    df = spark.table(table)
    print(f"\n### Tabela: {table} ###")

    for col in cols:
        if col in df.columns:
            print(f"\nFrequência dos valores da coluna '{col}':")
            freq_df = (
                df.groupBy(col)
                  .count()
                  .orderBy(F.desc("count"))
            )
            display(freq_df)



# ============================================================
# 4) MIN / MÁX DE COLUNAS NUMÉRICAS
# ============================================================

print("\n===== 4) MIN / MÁX DE COLUNAS NUMÉRICAS =====")

for table in tables:
    df = spark.table(table)

    numeric_cols = [
        c for c, t in df.dtypes 
        if t in ("int", "double", "float", "long", "bigint")
    ]

    if not numeric_cols:
        continue

    print(f"\n### Tabela: {table} ###")

    stats_df = df.select(
        *[F.min(c).alias(f"min_{c}") for c in numeric_cols],
        *[F.max(c).alias(f"max_{c}") for c in numeric_cols]
    )

    display(stats_df)



# ============================================================
# 5) HISTOGRAMAS — DISTRIBUIÇÃO
# ============================================================

print("\n===== 5) HISTOGRAMAS (DISTRIBUIÇÕES) =====")

# --- Minuto dos gols ---
print("\nHistograma – Distribuição dos Minutos dos Gols")

df_g = spark.table("silver_gols").select("minuto_int").dropna()
pdf_g = df_g.toPandas()

plt.figure()
plt.hist(pdf_g["minuto_int"], bins=20)
plt.title("Distribuição dos Minutos dos Gols")
plt.xlabel("Minuto")
plt.ylabel("Frequência")
plt.show()


# --- Posse de bola ---
print("\nHistograma – Distribuição da Posse de Bola (%)")

df_p = spark.table("silver_estatisticas").select("posse_de_bola_num").dropna()
pdf_p = df_p.toPandas()

plt.figure()
plt.hist(pdf_p["posse_de_bola_num"], bins=20)
plt.title("Distribuição da Posse de Bola (%)")
plt.xlabel("Posse de Bola (%)")
plt.ylabel("Frequência")
plt.show()



# ============================================================
# 6) VERIFICAÇÕES ESPECÍFICAS E DADOS AUSENTES
# ============================================================

print("\n===== 6) VERIFICAÇÕES ESPECÍFICAS E DADOS AUSENTES =====")

df_full = spark.table("silver_full")
total_full = df_full.count()
print(f"\nTotal de registros em silver_full: {total_full}")


# 6.1 – Datas realmente inválidas (se existirem)
print("\n6.1 – Registros com data_dt nula (esperado = 0):")
invalid_data_dt = df_full.filter("data_dt IS NULL")
display(invalid_data_dt)


# 6.2 – Horário indisponível (ausência na fonte)
print("\n6.2 – Registros sem horário válido (data_hora_ts nula):")
df_hora_null = df_full.filter("data_hora_ts IS NULL")
print(f"Registros sem horário válido: {df_hora_null.count()} de {total_full}")


# 6.3 – Colunas ausentes na origem
print("\n6.3 – Colunas de formação e técnico (dados ausentes na origem):")

cols_formacao_tecnico = [
    "formacao_mandante",
    "formacao_visitante",
    "tecnico_mandante",
    "tecnico_visitante"
]

for col_name in cols_formacao_tecnico:
    if col_name in df_full.columns:
        null_count = df_full.filter(F.col(col_name).isNull()).count()
        print(f"Coluna '{col_name}': {null_count} valores nulos")
    else:
        print(f"Coluna '{col_name}' não encontrada em silver_full.")


# 6.4 – Minutos inválidos
print("\n6.4 – Minutos de gol fora do intervalo 0–130 (esperado = 0):")
df_g = spark.table("silver_gols")
invalid_minute = df_g.filter("minuto_int < 0 OR minuto_int > 130")
display(invalid_minute)


# 6.5 – Posse de bola inválida
print("\n6.5 – Posse de bola fora do intervalo 0–100 (esperado = 0):")
df_e = spark.table("silver_estatisticas")
invalid_posse = df_e.filter("posse_de_bola_num < 0 OR posse_de_bola_num > 100")
display(invalid_posse)



# ============================================================
# 7) VERIFICAÇÃO DE UNICIDADE NA PRÓPRIA CAMADA SILVER
# (somente chaves nativas — sem dimensões)
# ============================================================

print("\n===== 7) VERIFICAÇÃO DE UNICIDADE (Silver) =====")

# silver_full — chave partida_id
print("\nUnicidade: silver_full.partida_id")
dup_partidas = (
    df_full.groupBy("partida_id").count().filter("count > 1")
)
display(dup_partidas)


print("\n=== FIM DA ANÁLISE DE QUALIDADE — CAMADA SILVER ===")